# SLAM 03 — Pose Graph Optimization: Closing the Loop

> **Geo-Instructor · SLAM Career Track · Notebook 3 of 3**

---

## What you will build

A robot navigates a corridor and returns to where it started. Odometry has drifted —
the map is broken. You will implement **pose graph optimization** to correct the entire
trajectory at once using the loop closure constraint.

### You will implement:
1. Pose graph construction (nodes = poses, edges = constraints)
2. Least-squares error formulation for each edge
3. Gradient descent optimizer (manual Jacobians)
4. scipy.optimize.minimize version (production-style)
5. Visualization of drift before/after optimization

---

## Why this matters for SLAM jobs

This is the **back-end** of every modern SLAM system:

| Front-end | Back-end |
|-----------|----------|
| ICP, feature tracking | Pose graph optimization |
| Provides: relative poses | Takes: all constraints, produces: optimal trajectory |
| EKF, visual odometry | g2o, GTSAM, Ceres Solver |

In production: **GTSAM** (used by Waymo, Apple) and **g2o** (used by ORB-SLAM) solve
exactly this problem using sparse Cholesky factorization.

Understanding pose graph optimization is the difference between a junior SLAM engineer
and a senior one.

---

```
Runtime: ~2 min  |  No GPU needed  |  numpy + scipy only
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import minimize
from scipy.sparse import lil_matrix
import warnings
warnings.filterwarnings('ignore')

np.random.seed(0)
plt.rcParams.update({
    'figure.facecolor': '#F4F2F0',
    'axes.facecolor':   '#F4F2F0',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.family': 'monospace',
})
print('Ready.')

---
## Part 1 — The Pose Graph Data Structure

A pose graph has:

- **Nodes** `x_i = [px_i, py_i, theta_i]` — robot poses (positions + headings)
- **Edges** `(i, j, z_ij, Omega_ij)` — constraints between poses:
  - `z_ij` = measured relative transform between pose i and pose j
  - `Omega_ij` = information matrix (inverse of covariance — how much we trust this edge)

```
  x_0 --odom-- x_1 --odom-- x_2 --odom-- ... --odom-- x_N
   |                                                    |
   +--------------- LOOP CLOSURE EDGE ─────────────────+
```

The loop closure edge says: "I recognize this place — my relative transform to x_0 is z".
Without optimization the graph is inconsistent (drift). With optimization we distribute
the error across the whole trajectory.

In [ ]:
# ── Simulate a robot traversing a loop ───────────────────────────────────────

def wrap_angle(a):
    """Wrap angle to [-pi, pi]."""
    return (a + np.pi) % (2 * np.pi) - np.pi

def compose_poses(x1, x2):
    """
    Compose two SE(2) poses.
    x1 = [px1, py1, theta1],  x2 = [px2, py2, theta2] (relative to x1)
    Returns absolute pose of x2 expressed globally.
    """
    px1, py1, th1 = x1
    px2, py2, th2 = x2
    c, s = np.cos(th1), np.sin(th1)
    px = px1 + c*px2 - s*py2
    py = py1 + s*px2 + c*py2
    th = wrap_angle(th1 + th2)
    return np.array([px, py, th])

def relative_pose(x1, x2):
    """Compute relative transform x2 expressed in frame of x1."""
    dx = x2[0] - x1[0]
    dy = x2[1] - x1[1]
    c, s = np.cos(x1[2]), np.sin(x1[2])
    rel_x = c*dx + s*dy
    rel_y = -s*dx + c*dy
    rel_th = wrap_angle(x2[2] - x1[2])
    return np.array([rel_x, rel_y, rel_th])

# ── Ground truth: square loop  ────────────────────────────────────────────────
N_per_side = 8
N = N_per_side * 4   # 32 poses total
side = 5.0

true_poses = []
# Go right
for i in range(N_per_side): true_poses.append([i * side/N_per_side, 0, 0])
# Go up
for i in range(N_per_side): true_poses.append([side, i * side/N_per_side, np.pi/2])
# Go left
for i in range(N_per_side): true_poses.append([side - i * side/N_per_side, side, np.pi])
# Go down
for i in range(N_per_side): true_poses.append([0, side - i * side/N_per_side, -np.pi/2])
true_poses = np.array(true_poses)

# ── Add cumulative odometry drift to get noisy observed poses ─────────────────
odom_sigma_trans  = 0.08   # m per step
odom_sigma_rot    = 0.04   # rad per step

noisy_poses = [true_poses[0].copy()]
for i in range(1, N):
    true_rel = relative_pose(true_poses[i-1], true_poses[i])
    noisy_rel = true_rel + np.array([
        np.random.randn() * odom_sigma_trans,
        np.random.randn() * odom_sigma_trans,
        np.random.randn() * odom_sigma_rot,
    ])
    noisy_poses.append(compose_poses(noisy_poses[-1], noisy_rel))
noisy_poses = np.array(noisy_poses)

print(f'Pose graph: {N} nodes')
print(f'Loop closure error before optimization:')
print(f'  position: {np.linalg.norm(noisy_poses[-1,:2] - noisy_poses[0,:2]):.3f}m')
print(f'  angle:    {np.degrees(abs(wrap_angle(noisy_poses[-1,2] - noisy_poses[0,2]))):.2f}deg')

In [ ]:
# ── Build the edge list ───────────────────────────────────────────────────────

class PoseGraph:
    def __init__(self):
        self.edges = []   # each: (i, j, z_ij, Omega_ij)

    def add_odometry_edge(self, i, j, z_ij, sigma_t=0.1, sigma_r=0.05):
        """Odometry constraints (sequential, high noise)."""
        Omega = np.diag([1/sigma_t**2, 1/sigma_t**2, 1/sigma_r**2])
        self.edges.append((i, j, z_ij, Omega))

    def add_loop_closure(self, i, j, z_ij, sigma_t=0.05, sigma_r=0.01):
        """Loop closure: highly trusted (measured carefully, e.g. by ICP)."""
        Omega = np.diag([1/sigma_t**2, 1/sigma_t**2, 1/sigma_r**2])
        self.edges.append((i, j, z_ij, Omega))


graph = PoseGraph()

# Sequential odometry edges (noisy)
for i in range(N - 1):
    z_ij = relative_pose(noisy_poses[i], noisy_poses[i+1])
    graph.add_odometry_edge(i, i+1, z_ij)

# Loop closure: node N-1 sees node 0 again (via ICP scan matching)
# Measure the relative pose from a fresh ICP run (assume near-perfect accuracy)
z_loop = relative_pose(true_poses[-1], true_poses[0])   # ground truth for demo
z_loop += np.random.randn(3) * [0.02, 0.02, 0.005]     # small ICP noise
graph.add_loop_closure(N-1, 0, z_loop)

print(f'Total edges: {len(graph.edges)}')
print(f'  - {N-1} odometry edges')
print(f'  - 1 loop closure edge')

---
## Part 2 — The Optimization Objective

For each edge `(i, j, z_ij, Omega_ij)` the **error** is:

```
  e_ij(x) = z_ij  -  relative_pose(x_i, x_j)
           = measurement  -  prediction
```

This is an "innovation" — exactly like the Kalman filter, but now batched over
the entire trajectory.

The **total cost** to minimize:

```
  F(x) = (1/2) sum_ij   e_ij(x).T * Omega_ij * e_ij(x)
```

This is **weighted least squares** — edges with higher information `Omega_ij`
(loop closures) exert more pull on the trajectory.

In [ ]:
# ── Cost function and gradient ────────────────────────────────────────────────

def edge_error(xi, xj, z_ij):
    """Error for a single edge: z_ij - relative_pose(xi, xj)."""
    z_hat = relative_pose(xi, xj)
    e = z_ij - z_hat
    e[2] = wrap_angle(e[2])
    return e


def total_cost(x_flat, graph):
    """
    x_flat: (3*N,) flattened pose array
    Returns scalar cost F(x).
    """
    n = len(x_flat) // 3
    poses = x_flat.reshape(n, 3)
    cost = 0.0
    for i, j, z_ij, Omega in graph.edges:
        e = edge_error(poses[i], poses[j], z_ij)
        cost += 0.5 * e @ Omega @ e
    return cost


def total_gradient(x_flat, graph):
    """
    Compute gradient dF/dx numerically (finite differences).
    For production use analytic Jacobians; this demonstrates the concept.
    """
    grad = np.zeros_like(x_flat)
    eps = 1e-6
    f0 = total_cost(x_flat, graph)
    for k in range(len(x_flat)):
        x_plus = x_flat.copy(); x_plus[k] += eps
        grad[k] = (total_cost(x_plus, graph) - f0) / eps
    return grad


# Evaluate initial cost
x0_flat = noisy_poses.flatten()
cost_initial = total_cost(x0_flat, graph)
print(f'Initial cost: {cost_initial:.3f}')
print('(Finite-difference gradient ready for optimization.)')

In [ ]:
# ── Analytic Jacobian of the edge error function ──────────────────────────────
# In production SLAM (g2o, GTSAM, Ceres) these are computed analytically.
# Here we implement them for the 2-D case.

def edge_jacobians(xi, xj):
    """
    Jacobians of relative_pose(xi, xj) w.r.t. xi and xj.
    Returns Ji (3x3), Jj (3x3).
    """
    dx = xj[0] - xi[0]
    dy = xj[1] - xi[1]
    ci, si = np.cos(xi[2]), np.sin(xi[2])

    # d(relative_pose) / d(xi)
    Ji = np.array([
        [-ci, -si,  -si*dx + ci*dy],
        [ si, -ci,  -ci*dx - si*dy],
        [ 0,   0,   -1            ],
    ])

    # d(relative_pose) / d(xj)
    Jj = np.array([
        [ ci,  si,  0],
        [-si,  ci,  0],
        [ 0,   0,   1],
    ])

    return Ji, Jj


def total_gradient_analytic(x_flat, graph):
    """Gradient using analytic Jacobians — much faster than finite differences."""
    n = len(x_flat) // 3
    poses = x_flat.reshape(n, 3)
    grad = np.zeros_like(x_flat)

    for i, j, z_ij, Omega in graph.edges:
        xi, xj = poses[i], poses[j]
        e = edge_error(xi, xj, z_ij)    # shape (3,)
        Ji, Jj = edge_jacobians(xi, xj)

        # Gradient contribution: -J.T * Omega * e
        gi = -Ji.T @ Omega @ e
        gj = -Jj.T @ Omega @ e

        grad[i*3:(i+1)*3] += gi
        grad[j*3:(j+1)*3] += gj

    # Fix pose 0 (gauge freedom — the absolute reference must be anchored)
    grad[:3] = 0

    return grad


# Verify: analytic vs numerical gradient
g_num     = total_gradient(x0_flat, graph)
g_analytic = total_gradient_analytic(x0_flat, graph)
max_diff  = np.max(np.abs(g_num - g_analytic))
print(f'Max |analytic - numerical| gradient: {max_diff:.2e}')
print('(Should be < 1e-4 for a correct Jacobian)')

---
## Part 3 — Optimizing the Pose Graph

We use `scipy.optimize.minimize` with L-BFGS-B (a quasi-Newton method).
In production SLAM:
- **g2o** uses Levenberg-Marquardt or Gauss-Newton with a sparse Cholesky solve
- **GTSAM** uses Bayes tree / iSAM2 for incremental updates
- **Ceres Solver** uses automatic differentiation + trust-region methods

All are solving the same weighted least-squares problem we defined above.

In [ ]:
# ── Run pose graph optimization ───────────────────────────────────────────────
cost_history = []

def cost_with_log(x_flat):
    c = total_cost(x_flat, graph)
    cost_history.append(c)
    return c

result = minimize(
    cost_with_log,
    x0_flat.copy(),
    jac=lambda x: total_gradient_analytic(x, graph),
    method='L-BFGS-B',
    options={'maxiter': 500, 'ftol': 1e-12, 'gtol': 1e-8},
)

optimized_poses = result.x.reshape(N, 3)

print(f'Converged: {result.success}  |  message: {result.message}')
print(f'Cost:  {cost_initial:.3f}  ->  {result.fun:.5f}  (reduced {cost_initial/result.fun:.0f}x)')
print()
print('Loop closure error AFTER optimization:')
print(f'  position: {np.linalg.norm(optimized_poses[-1,:2] - optimized_poses[0,:2]):.4f}m')
print(f'  angle:    {np.degrees(abs(wrap_angle(optimized_poses[-1,2] - optimized_poses[0,2]))):.3f}deg')

In [ ]:
# ── Compare trajectories ──────────────────────────────────────────────────────

def plot_trajectory(ax, poses, color, label, alpha=1.0, lw=2, show_poses=True):
    ax.plot(poses[:,0], poses[:,1], '-', color=color, lw=lw, alpha=alpha, label=label)
    if show_poses:
        ax.scatter(poses[:,0], poses[:,1], s=20, c=color, zorder=5, alpha=alpha)
    # Draw heading arrows at every 4th pose
    for i in range(0, len(poses), 4):
        ax.annotate('', xy=(poses[i,0]+0.3*np.cos(poses[i,2]),
                             poses[i,1]+0.3*np.sin(poses[i,2])),
                    xytext=(poses[i,0], poses[i,1]),
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.2, alpha=alpha))


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Pose Graph Optimization: Before vs After', fontsize=12)

# --- Before ---
ax = axes[0]
plot_trajectory(ax, true_poses,  '#2A2622', 'Ground truth', alpha=0.4, lw=3)
plot_trajectory(ax, noisy_poses, 'tomato',  'Noisy odometry (drift)', lw=2)
# Show loop closure gap
ax.annotate('', xy=noisy_poses[0,:2], xytext=noisy_poses[-1,:2],
            arrowprops=dict(arrowstyle='->', color='purple', lw=2))
ax.text(0.5, 0.02, 'Loop closure gap', transform=ax.transAxes,
        ha='center', color='purple', fontsize=8)
ax.set_aspect('equal'); ax.legend(fontsize=8)
ax.set_title('Before optimization\n(drift visible)')

# --- After ---
ax = axes[1]
plot_trajectory(ax, true_poses,      '#2A2622',  'Ground truth', alpha=0.4, lw=3)
plot_trajectory(ax, noisy_poses,     'tomato',   'Before (noisy)', alpha=0.3, lw=1.5, show_poses=False)
plot_trajectory(ax, optimized_poses, 'steelblue','After optimization', lw=2)
ax.set_aspect('equal'); ax.legend(fontsize=8)
ax.set_title('After optimization\n(loop closed)')

# --- Error comparison ---
ax = axes[2]
pose_errs_before = np.linalg.norm(noisy_poses[:,:2]     - true_poses[:,:2], axis=1)
pose_errs_after  = np.linalg.norm(optimized_poses[:,:2] - true_poses[:,:2], axis=1)

t_idx = np.arange(N)
ax.plot(t_idx, pose_errs_before, 'tomato',    lw=2, label=f'Before (mean={pose_errs_before.mean():.3f}m)')
ax.plot(t_idx, pose_errs_after,  'steelblue', lw=2, label=f'After  (mean={pose_errs_after.mean():.3f}m)')
ax.set_xlabel('Pose index'); ax.set_ylabel('Position error (m)')
ax.set_title('Per-pose error before/after optimization')
ax.legend(fontsize=8)
ax.set_ylim(bottom=0)

plt.tight_layout(); plt.show()

improvement = pose_errs_before.mean() / pose_errs_after.mean()
print(f'Mean position error: {pose_errs_before.mean():.3f}m -> {pose_errs_after.mean():.3f}m  ({improvement:.1f}x better)')

In [ ]:
# ── Optimization convergence ──────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

ax1.semilogy(cost_history, 'steelblue', lw=2)
ax1.set_xlabel('Function evaluation count')
ax1.set_ylabel('Total cost F(x) — log scale')
ax1.set_title('Cost convergence during L-BFGS-B optimization')

# Information matrix visualization
# Show which nodes are connected to which (adjacency)
adj = np.zeros((N, N))
weights = np.zeros((N, N))
for i, j, z_ij, Omega in graph.edges:
    adj[i, j] = 1; adj[j, i] = 1
    w = np.trace(Omega) / 3  # mean information
    weights[i, j] = w; weights[j, i] = w

im = ax2.imshow(weights, cmap='hot_r', aspect='auto')
plt.colorbar(im, ax=ax2, label='Mean information (Omega trace / 3)')
ax2.set_xlabel('Pose j'); ax2.set_ylabel('Pose i')
ax2.set_title('Information matrix structure\n(bright = high-confidence edges)')

plt.tight_layout(); plt.show()
print('The loop closure edge (0, N-1) has highest information -> strongest constraint.')

---
## Part 4 — What Happens with Multiple Loop Closures?

Real SLAM systems detect **many** loop closures over time (every time the robot
revisits a location). Each one adds a constraint to the pose graph.

More loop closures = better-constrained problem = lower drift.

In [ ]:
# ── Test: 0, 1, 2, 4 loop closures ───────────────────────────────────────────

def optimize_with_n_loops(n_loops):
    """Add n evenly-spaced loop closure edges and optimize."""
    g = PoseGraph()
    for i in range(N - 1):
        z_ij = relative_pose(noisy_poses[i], noisy_poses[i+1])
        g.add_odometry_edge(i, i+1, z_ij)

    # Add loop closures at evenly-spaced indices
    if n_loops > 0:
        loop_anchors = np.linspace(0, N-1, n_loops+1, dtype=int)[1:]  # skip start
        for lc_idx in loop_anchors:
            # Find nearest true pose to current noisy pose[lc_idx]
            anchor = lc_idx % N
            # Simulate ICP giving near-perfect relative pose
            z_lc = relative_pose(true_poses[anchor], true_poses[0])
            z_lc += np.random.randn(3) * [0.02, 0.02, 0.005]
            g.add_loop_closure(anchor, 0, z_lc)

    result = minimize(
        lambda x: total_cost(x, g),
        noisy_poses.flatten(),
        jac=lambda x: total_gradient_analytic(x, g),
        method='L-BFGS-B',
        options={'maxiter': 500, 'ftol': 1e-12, 'gtol': 1e-8},
    )
    opt = result.x.reshape(N, 3)
    err = np.linalg.norm(opt[:,:2] - true_poses[:,:2], axis=1).mean()
    return opt, err

n_loops_list = [0, 1, 2, 4]
results = {}
print('Loop closures -> mean position error:')
for nl in n_loops_list:
    opt, err = optimize_with_n_loops(nl)
    results[nl] = (opt, err)
    print(f'  {nl} loop closures:  {err:.4f} m')

fig, axes = plt.subplots(1, len(n_loops_list), figsize=(18, 4.5))
for ax, nl in zip(axes, n_loops_list):
    opt, err = results[nl]
    ax.plot(true_poses[:,0], true_poses[:,1], 'k-', lw=3, alpha=0.3, label='GT')
    ax.plot(opt[:,0], opt[:,1], 'steelblue', lw=2)
    ax.scatter(opt[:,0], opt[:,1], s=15, c='steelblue', zorder=5)
    ax.set_aspect('equal')
    ax.set_title(f'{nl} loop closure{"s" if nl!=1 else ""}\nmean err={err:.3f}m', fontsize=9)
    ax.set_xlim(-1, 7); ax.set_ylim(-1, 7)
plt.suptitle('More loop closures = less drift', fontsize=11)
plt.tight_layout(); plt.show()

---
## Part 5 — Exercises

### Exercise 1 — Bad loop closure (outlier)
Corrupt one loop closure edge with large noise (`sigma=1.0m`).  
What happens to the optimized trajectory?  
Now implement **switchable constraints** or **Cauchy robust cost** to downweight outliers.

### Exercise 2 — Incremental optimization (iSAM2 style)
Instead of batch optimization, implement an incremental version:
after each new pose, run 3 iterations of gradient descent (not full convergence).
Watch the real-time map update as new poses arrive.

### Exercise 3 — Use GTSAM
Install `gtsam` (`pip install gtsam`) and recreate this exact graph using
`gtsam.BetweenFactorPose2`. Compare solution quality and speed.

### Exercise 4 — 3-D pose graph
Extend to SE(3): each node is `[x, y, z, roll, pitch, yaw]` (6-DOF).  
The Jacobian becomes 6x6. This is the setting for full 3-D LiDAR SLAM.

---
## Full SLAM Pipeline Summary

```
  SENSOR INPUT
    LiDAR / Camera / IMU
         |
  FRONT-END              (Notebook 1 + 2)
    EKF odometry
    ICP scan matching
    Feature extraction
    Loop closure detection
         |
  POSE GRAPH             (This notebook)
    Nodes = poses
    Edges = constraints
         |
  BACK-END OPTIMIZATION  (What GTSAM/g2o/Ceres do)
    Gauss-Newton / L-BFGS
    Sparse Cholesky
    Incremental update (iSAM2)
         |
  OUTPUT: Optimal trajectory + 3-D map
```

## Key Libraries in Production SLAM

| Library | Language | Used by |
|---------|----------|--------|
| GTSAM | C++ / Python | Waymo, Apple Maps, MIT |
| g2o | C++ | ORB-SLAM2/3, LSD-SLAM |
| Ceres Solver | C++ | Google Street View, Cartographer |
| Open3D | C++ / Python | Quick prototyping |
| Factor Graph | Any | Conceptual model used everywhere |

**You have now built the entire core of a 2-D SLAM system from first principles.**